# Unit II — Fundamentals of Quantum Mechanics for Computing

This notebook introduces the core ideas needed to understand quantum computing:

- Classical bits and qubits
- Quantum state vectors
- Probability amplitudes
- Measurement
- Superposition
- Quantum interference
- Multiple qubits
- Tensor products
- Entanglement
- Wave-particle duality at a conceptual level

The examples use basic Python, NumPy, and Matplotlib. No IBM account or API key is required.

In [ ]:
# Install the required libraries when needed.
# Remove the first # and run the command in a new environment.

# %pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

## 1. Classical Bit and Qubit

A classical bit is always in one definite state:

\[
0 \quad \text{or} \quad 1
\]

A qubit is represented using a state vector:

\[
|\psi\rangle = \alpha|0\rangle + \beta|1\rangle
\]

Here:

- \(\alpha\) is the probability amplitude of \(|0\rangle\)
- \(\beta\) is the probability amplitude of \(|1\rangle\)
- The amplitudes may be real or complex numbers
- The probabilities are \(|\alpha|^2\) and \(|\beta|^2\)

A valid qubit must satisfy:

\[
|\alpha|^2 + |\beta|^2 = 1
\]

In [ ]:
# Computational basis states

ket_0 = np.array([1, 0], dtype=complex)
ket_1 = np.array([0, 1], dtype=complex)

print("|0> =", ket_0)
print("|1> =", ket_1)

## 2. Checking Whether a State Is Valid

The sum of the measurement probabilities must be equal to 1.

In [ ]:
def state_probabilities(state):
    """Return measurement probabilities for a quantum state vector."""
    return np.abs(state) ** 2


def is_normalized(state, tolerance=1e-10):
    """Check whether the total probability is approximately 1."""
    total_probability = np.sum(state_probabilities(state))
    return np.isclose(total_probability, 1.0, atol=tolerance)


print("Is |0> normalized?", is_normalized(ket_0))
print("Is |1> normalized?", is_normalized(ket_1))

## 3. Creating a Superposition State

An equal superposition of \(|0\rangle\) and \(|1\rangle\) is:

\[
|+\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}
\]

Its amplitudes are both \(1/\sqrt{2}\), so each measurement probability is \(1/2\).

In [ ]:
plus_state = (ket_0 + ket_1) / np.sqrt(2)

print("|+> state:", plus_state)
print("Probabilities:", state_probabilities(plus_state))
print("Normalized:", is_normalized(plus_state))

Another important superposition is:

\[
|-\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}}
\]

The \(|+\rangle\) and \(|-\rangle\) states have the same measurement probabilities in the computational basis, but their relative phases are different.

In [ ]:
minus_state = (ket_0 - ket_1) / np.sqrt(2)

print("|-> state:", minus_state)
print("Probabilities:", state_probabilities(minus_state))
print("Normalized:", is_normalized(minus_state))

## 4. Displaying Measurement Probabilities

In [ ]:
def plot_probabilities(state, labels=None, title="Measurement probabilities"):
    probabilities = state_probabilities(state)

    if labels is None:
        labels = [f"|{index}>" for index in range(len(state))]

    plt.figure(figsize=(6, 4))
    plt.bar(labels, probabilities)
    plt.ylim(0, 1)
    plt.xlabel("Measurement result")
    plt.ylabel("Probability")
    plt.title(title)
    plt.show()


plot_probabilities(
    plus_state,
    labels=["|0>", "|1>"],
    title="Equal superposition"
)

## 5. General Single-Qubit State

A convenient parameterization of a single-qubit state is:

\[
|\psi\rangle =
\cos\left(\frac{\theta}{2}\right)|0\rangle
+
e^{i\phi}\sin\left(\frac{\theta}{2}\right)|1\rangle
\]

The angles \(\theta\) and \(\phi\) describe a point on the Bloch sphere.

In [ ]:
def create_qubit(theta, phi):
    """Create a normalized single-qubit state from theta and phi."""
    alpha = np.cos(theta / 2)
    beta = np.exp(1j * phi) * np.sin(theta / 2)
    return np.array([alpha, beta], dtype=complex)


theta = np.pi / 3
phi = np.pi / 4

psi = create_qubit(theta, phi)

print("State vector:", psi)
print("Probabilities:", state_probabilities(psi))
print("Normalized:", is_normalized(psi))

## 6. Global Phase and Relative Phase

Multiplying the whole state by the same phase factor does not change its measurement probabilities.

For example:

\[
|\psi'\rangle = e^{i\gamma}|\psi\rangle
\]

The relative phase between amplitudes, however, can affect interference.

In [ ]:
gamma = np.pi / 3
phase_shifted_state = np.exp(1j * gamma) * plus_state

print("Original state:", plus_state)
print("Phase-shifted state:", phase_shifted_state)

print("\nOriginal probabilities:", state_probabilities(plus_state))
print("Phase-shifted probabilities:", state_probabilities(phase_shifted_state))

## 7. Measurement Simulation

Measurement converts a quantum state into a classical result.

For a qubit:

- Result `0` occurs with probability \(|\alpha|^2\)
- Result `1` occurs with probability \(|\beta|^2\)

Repeated measurements reveal the probability distribution.

In [ ]:
def measure_state(state, shots=1, seed=None):
    """Simulate repeated measurements of a normalized state vector."""
    if not is_normalized(state):
        raise ValueError("The state vector must be normalized.")

    rng = np.random.default_rng(seed)
    probabilities = state_probabilities(state)
    outcomes = np.arange(len(state))

    return rng.choice(outcomes, size=shots, p=probabilities)


results = measure_state(plus_state, shots=20, seed=42)
print("Measurement results:", results)

In [ ]:
def measurement_counts(state, shots=1000, seed=None):
    results = measure_state(state, shots=shots, seed=seed)
    unique, counts = np.unique(results, return_counts=True)
    return {int(key): int(value) for key, value in zip(unique, counts)}


counts = measurement_counts(plus_state, shots=1000, seed=42)
print("Counts:", counts)

In [ ]:
shots = 1000
counts = measurement_counts(plus_state, shots=shots, seed=42)

labels = ["0", "1"]
frequencies = [counts.get(0, 0), counts.get(1, 0)]

plt.figure(figsize=(6, 4))
plt.bar(labels, frequencies)
plt.xlabel("Measurement result")
plt.ylabel("Number of occurrences")
plt.title(f"Simulated measurement of |+> using {shots} shots")
plt.show()

## 8. The Hadamard Transformation

The Hadamard matrix is:

\[
H = \frac{1}{\sqrt{2}}
\begin{bmatrix}
1 & 1\\
1 & -1
\end{bmatrix}
\]

It transforms basis states into superposition states:

\[
H|0\rangle = |+\rangle
\]

\[
H|1\rangle = |-\rangle
\]

In [ ]:
H = (1 / np.sqrt(2)) * np.array(
    [
        [1, 1],
        [1, -1]
    ],
    dtype=complex
)

print("H|0> =", H @ ket_0)
print("H|1> =", H @ ket_1)

## 9. Quantum Interference

Quantum amplitudes can add constructively or destructively.

Applying Hadamard twice gives:

\[
H(H|0\rangle) = |0\rangle
\]

The first Hadamard creates a superposition. The second recombines the amplitudes so that one outcome is strengthened and the other is cancelled.

In [ ]:
state_after_first_h = H @ ket_0
state_after_second_h = H @ state_after_first_h

print("Initial state       :", ket_0)
print("After first H       :", state_after_first_h)
print("After second H      :", state_after_second_h)
print("Final probabilities :", state_probabilities(state_after_second_h))

### Destructive Interference Example

For the \(|1\rangle\) output amplitude:

\[
\frac{1}{2} - \frac{1}{2} = 0
\]

The two contributions cancel.

In [ ]:
path_1_amplitude = 1 / 2
path_2_amplitude = -1 / 2

combined_amplitude = path_1_amplitude + path_2_amplitude
combined_probability = abs(combined_amplitude) ** 2

print("Path 1 amplitude:", path_1_amplitude)
print("Path 2 amplitude:", path_2_amplitude)
print("Combined amplitude:", combined_amplitude)
print("Probability:", combined_probability)

### Constructive Interference Example

When amplitudes have the same sign, they reinforce each other.

In [ ]:
path_1_amplitude = 1 / 2
path_2_amplitude = 1 / 2

combined_amplitude = path_1_amplitude + path_2_amplitude
combined_probability = abs(combined_amplitude) ** 2

print("Combined amplitude:", combined_amplitude)
print("Probability:", combined_probability)

## 10. Visualizing Interference with Waves

The following example uses ordinary sine waves to illustrate constructive and destructive interference. It is a conceptual analogy for combining amplitudes.

In [ ]:
x = np.linspace(0, 4 * np.pi, 500)

wave_1 = np.sin(x)
wave_2_same_phase = np.sin(x)
constructive_result = wave_1 + wave_2_same_phase

plt.figure(figsize=(9, 5))
plt.plot(x, wave_1, label="Wave 1")
plt.plot(x, wave_2_same_phase, label="Wave 2")
plt.plot(x, constructive_result, label="Combined wave")
plt.xlabel("Position or time")
plt.ylabel("Amplitude")
plt.title("Constructive interference")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
wave_2_opposite_phase = np.sin(x + np.pi)
destructive_result = wave_1 + wave_2_opposite_phase

plt.figure(figsize=(9, 5))
plt.plot(x, wave_1, label="Wave 1")
plt.plot(x, wave_2_opposite_phase, label="Wave 2")
plt.plot(x, destructive_result, label="Combined wave")
plt.xlabel("Position or time")
plt.ylabel("Amplitude")
plt.title("Destructive interference")
plt.grid(True)
plt.legend()
plt.show()

## 11. Two-Qubit Basis States

Two qubits have four computational basis states:

\[
|00\rangle,\ |01\rangle,\ |10\rangle,\ |11\rangle
\]

They are constructed using the tensor product.

In [ ]:
ket_00 = np.kron(ket_0, ket_0)
ket_01 = np.kron(ket_0, ket_1)
ket_10 = np.kron(ket_1, ket_0)
ket_11 = np.kron(ket_1, ket_1)

print("|00> =", ket_00)
print("|01> =", ket_01)
print("|10> =", ket_10)
print("|11> =", ket_11)

## 12. Tensor Product of Qubits

Suppose the first qubit is \(|+\rangle\) and the second is \(|0\rangle\).

Their joint state is:

\[
|+\rangle \otimes |0\rangle
=
\frac{|00\rangle + |10\rangle}{\sqrt{2}}
\]

In [ ]:
joint_state = np.kron(plus_state, ket_0)

print("Joint state:", joint_state)
print("Probabilities:", state_probabilities(joint_state))

plot_probabilities(
    joint_state,
    labels=["|00>", "|01>", "|10>", "|11>"],
    title="Joint state |+> tensor |0>"
)

## 13. Entanglement

An entangled state cannot be written as a simple tensor product of independent single-qubit states.

A standard example is the Bell state:

\[
|\Phi^+\rangle =
\frac{|00\rangle + |11\rangle}{\sqrt{2}}
\]

Measurement produces:

- `00` with probability \(1/2\)
- `11` with probability \(1/2\)
- `01` and `10` with probability \(0\)

The two measurement results are correlated.

In [ ]:
bell_state = (ket_00 + ket_11) / np.sqrt(2)

print("Bell state:", bell_state)
print("Probabilities:", state_probabilities(bell_state))
print("Normalized:", is_normalized(bell_state))

plot_probabilities(
    bell_state,
    labels=["|00>", "|01>", "|10>", "|11>"],
    title="Bell-state measurement probabilities"
)

In [ ]:
bell_counts = measurement_counts(bell_state, shots=1000, seed=42)

basis_labels = ["00", "01", "10", "11"]
basis_counts = [bell_counts.get(index, 0) for index in range(4)]

plt.figure(figsize=(7, 4))
plt.bar(basis_labels, basis_counts)
plt.xlabel("Two-qubit measurement result")
plt.ylabel("Number of occurrences")
plt.title("Simulated Bell-state measurements")
plt.show()

print("Counts:", dict(zip(basis_labels, basis_counts)))

## 14. Product State Versus Entangled State

A two-qubit product state can be expressed as:

\[
|\psi\rangle \otimes |\phi\rangle
\]

For a state with amplitudes \([a,b,c,d]\), a simple test for separability is:

\[
ad - bc = 0
\]

For a pure two-qubit state, a nonzero value indicates entanglement.

In [ ]:
def separability_value(two_qubit_state):
    """Return ad - bc for a two-qubit pure state [a, b, c, d]."""
    if len(two_qubit_state) != 4:
        raise ValueError("A two-qubit state must have four amplitudes.")

    a, b, c, d = two_qubit_state
    return a * d - b * c


product_state = np.kron(plus_state, ket_0)

print("Product-state test value:", separability_value(product_state))
print("Bell-state test value   :", separability_value(bell_state))

## 15. Classical Correlation and Quantum Entanglement

Classical correlation and entanglement are not identical.

A classical system may randomly produce either `00` or `11`. That gives correlated outcomes, but it does not contain a coherent superposition.

The Bell state contains amplitudes for both \(|00\rangle\) and \(|11\rangle\), including a definite relative phase. This phase can influence later interference experiments.

## 16. Complex Probability Amplitudes

Quantum amplitudes can be complex numbers.

Consider:

\[
|\psi\rangle =
\frac{|0\rangle + i|1\rangle}{\sqrt{2}}
\]

The amplitudes differ in phase, but both measurement probabilities are still \(1/2\).

In [ ]:
complex_state = np.array(
    [1 / np.sqrt(2), 1j / np.sqrt(2)],
    dtype=complex
)

print("State:", complex_state)
print("Probabilities:", state_probabilities(complex_state))
print("Normalized:", is_normalized(complex_state))

## 17. Wave-Particle Duality — Conceptual Demonstration

Quantum objects can display particle-like detection events and wave-like interference patterns.

A simplified double-slit intensity model is:

\[
I(x) \propto \cos^2(kx)
\]

Individual detections are discrete, but many detections gradually form an interference distribution.

The simulation below is only a conceptual probability model.

In [ ]:
positions = np.linspace(-5, 5, 1000)

# Simplified interference pattern under a Gaussian envelope
envelope = np.exp(-(positions ** 2) / 8)
interference = np.cos(4 * positions) ** 2
intensity = envelope * interference
intensity = intensity / np.max(intensity)

plt.figure(figsize=(9, 4))
plt.plot(positions, intensity)
plt.xlabel("Screen position")
plt.ylabel("Relative detection probability")
plt.title("Simplified interference probability pattern")
plt.grid(True)
plt.show()

In [ ]:
# Generate particle-like detection positions from the wave-like probability pattern.

probability_distribution = intensity / np.sum(intensity)
rng = np.random.default_rng(42)

detected_positions = rng.choice(
    positions,
    size=3000,
    p=probability_distribution
)

plt.figure(figsize=(9, 4))
plt.hist(detected_positions, bins=100)
plt.xlabel("Detected screen position")
plt.ylabel("Detection count")
plt.title("Discrete detections forming an interference pattern")
plt.show()

## 18. Comparison of Classical Bits and Qubits

| Feature | Classical bit | Qubit |
|---|---|---|
| Basic states | `0` or `1` | \(|0\rangle\) and \(|1\rangle\) |
| General state | One definite value | Superposition of basis states |
| Description | Boolean value | Complex state vector |
| Reading | Reveals stored bit | Probabilistic measurement |
| Multiple units | Bit strings | Tensor-product state space |
| Correlation | Classical correlation | May include entanglement |
| Operations | Logic gates | Unitary quantum gates |
| Copying | Can usually be copied | Unknown states cannot be perfectly cloned |

## 19. Important Observations

1. A qubit is not merely a classical bit with an unknown value.
2. Probability amplitudes, not ordinary probabilities, are transformed by quantum gates.
3. Amplitudes may be negative or complex.
4. Measurement probabilities are obtained by taking squared magnitudes.
5. Relative phase enables interference.
6. Entanglement creates correlations that cannot be described as independent qubit states.
7. Quantum measurement produces classical outcomes.

## 20. Practice Exercises

1. Create the state  
   \[
   \frac{\sqrt{3}}{2}|0\rangle + \frac{1}{2}|1\rangle
   \]
   and verify its normalization.

2. Calculate its measurement probabilities.

3. Simulate 2,000 measurements of the state.

4. Apply the Hadamard matrix to \(|1\rangle\).

5. Apply Hadamard twice to \(|1\rangle\) and check the result.

6. Construct the two-qubit state  
   \[
   \frac{|01\rangle + |10\rangle}{\sqrt{2}}
   \]

7. Plot the measurement probabilities of that state.

8. Use the separability test to check whether it is entangled.

9. Change the phase in the wave-interference example and observe the pattern.

10. Explain the difference between a probability amplitude and a probability.

In [ ]:
# Practice area

# Write your code below.


## Summary

This notebook introduced:

- Qubits and state vectors
- Basis states \(|0\rangle\) and \(|1\rangle\)
- Normalization
- Probability amplitudes
- Measurement probabilities
- Superposition
- Phase
- Hadamard transformation
- Constructive and destructive interference
- Tensor products
- Two-qubit states
- Bell-state entanglement
- Wave-particle duality through a conceptual simulation

These ideas form the foundation for quantum gates and circuits in the next unit.